# 4.2. Краевая задача

Реализация: `lab4/bvp.py` (стрельба, конечные разности). Данные — из `data/lab4/4_2/`.

$$x(x-1)y'' - x y' + y = 0,\quad y(1)=2,\quad 2y'(2)-y(2)=1$$

Точное: $y(x)=1+x+x\ln|x|$, отрезок $[1,2]$, $h=0.1$.

Коэффициент при $y''$ обращается в ноль при $x=1$; стрельба ведётся справа с экстраполяцией левого условия.

In [1]:
import sys
import math
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "lab4"

from lab4.dataio import read_interval, read_scalar, read_vector, read_bc
from lab4.helpers import runge_romberg, fmt_row
from lab4.ode_ivp import euler, rk4, adams4, second_order_to_system
from lab4.bvp import shooting, finite_difference, linear_rhs_from_coefs

a, b = read_interval(DATA / "4_2" / "interval.txt")
h = read_scalar(DATA / "4_2" / "h.txt")
bc_left = read_bc(DATA / "4_2" / "bc_left.txt")
bc_right = read_bc(DATA / "4_2" / "bc_right.txt")
print(f"отрезок [{a}, {b}], h = {h}")
print("BC left α,β,γ =", bc_left)
print("BC right α,β,γ =", bc_right)

отрезок [1.0, 2.0], h = 0.1
BC left α,β,γ = (1.0, 0.0, 2.0)
BC right α,β,γ = (-1.0, 2.0, 1.0)


## Коэффициенты и точное решение

In [2]:
def a_coef(x):
    return x * (x - 1.0)

def b_coef(x):
    return -x

def c_coef(x):
    return 1.0

def d_coef(x):
    return 0.0

def exact(x):
    return 1.0 + x + x * math.log(abs(x))

rhs = linear_rhs_from_coefs(a_coef, b_coef, c_coef, d_coef)

## Метод стрельбы

In [3]:
xs_s, ys_s, eta, resid = shooting(
    rhs, a, b, h, bc_left, bc_right,
    eta0=4.0, eta1=5.0, from_right=True, singular_left=True,
)
print(f"η = y(b) = {eta}, φ(η) = {resid}")
print(f"{'x':>8} {'shooting':>14} {'exact':>14} {'error':>14}")
for i, x in enumerate(xs_s):
    ye = exact(x)
    print(fmt_row([x, ys_s[i][0], ye, ys_s[i][0] - ye]))
print("max |y - exact| =", max(abs(ys_s[i][0] - exact(xs_s[i])) for i in range(len(xs_s))))

η = y(b) = 4.3865843055001035, φ(η) = 4.440892098500626e-16
       x       shooting          exact          error
             1              2              2              0
           1.1      2.2049928      2.2048412  0.00015161501
           1.2      2.4189569      2.4187859  0.00017102138
           1.3      2.6412605      2.6410735  0.00018692499
           1.4      2.8712632      2.8710611  0.00020203754
           1.5      3.1084145      3.1081977  0.00021686757
           1.6      3.3522374      3.3520058  0.00023157022
           1.7      3.6023142       3.602068  0.00024620684
           1.8      3.8582768       3.858016  0.00026080595
           1.9      4.1197978      4.1195224  0.00027538235
             2      4.3865843      4.3862944  0.00028994438
max |y - exact| = 0.0002899443802126811


## Конечно-разностный метод

In [4]:
xs_f, y_f = finite_difference(
    a_coef, b_coef, c_coef, d_coef, a, b, h, bc_left, bc_right,
)
print(f"{'x':>8} {'FD':>14} {'exact':>14} {'error':>14}")
for i, x in enumerate(xs_f):
    ye = exact(x)
    print(fmt_row([x, y_f[i], ye, y_f[i] - ye]))
print("max |y - exact| =", max(abs(y_f[i] - exact(xs_f[i])) for i in range(len(xs_f))))

       x             FD          exact          error
             1              2              2              0
           1.1      2.2049869      2.2048412  0.00014567886
           1.2      2.4190408      2.4187859  0.00025493357
           1.3      2.6414062      2.6410735  0.00033264406
           1.4      2.8714437      2.8710611  0.00038256065
           1.5      3.1086053      3.1081977  0.00040764329
           1.6      3.3524161      3.3520058  0.00041028106
           1.7      3.6024605       3.602068  0.00039243924
           1.8      3.8583718       3.858016  0.00035576111
           1.9       4.119824      4.1195224  0.00030164014
             2      4.3865256      4.3862944  0.00023127251
max |y - exact| = 0.0004102810559798442


## Рунге–Ромберг

In [5]:
xs1, ys1, _, _ = shooting(
    rhs, a, b, h, bc_left, bc_right,
    eta0=4.0, eta1=5.0, from_right=True, singular_left=True,
)
xs2, ys2, _, _ = shooting(
    rhs, a, b, 2 * h, bc_left, bc_right,
    eta0=4.0, eta1=5.0, from_right=True, singular_left=True,
)
yf = [row[0] for row in ys1]
yc = [row[0] for row in ys2]
ref, R = runge_romberg(yf, yc, 4)
print("=== shooting, p=4 ===")
print(f"{'x':>8} {'y(h)':>14} {'y(2h)':>14} {'R':>14} {'y_RR':>14} {'exact':>14}")
for i, x in enumerate(xs2):
    print(fmt_row([x, yf[2 * i], yc[i], R[i], ref[i], exact(x)]))
print("max |R| =", max(abs(r) for r in R))
print("max |y_RR - exact| =", max(abs(ref[i] - exact(xs2[i])) for i in range(len(xs2))))
print()

xs1, y1 = finite_difference(a_coef, b_coef, c_coef, d_coef, a, b, h, bc_left, bc_right)
xs2, y2 = finite_difference(a_coef, b_coef, c_coef, d_coef, a, b, 2 * h, bc_left, bc_right)
ref, R = runge_romberg(y1, y2, 2)
print("=== finite differences, p=2 ===")
print(f"{'x':>8} {'y(h)':>14} {'y(2h)':>14} {'R':>14} {'y_RR':>14} {'exact':>14}")
for i, x in enumerate(xs2):
    print(fmt_row([x, y1[2 * i], y2[i], R[i], ref[i], exact(x)]))
print("max |R| =", max(abs(r) for r in R))
print("max |y_RR - exact| =", max(abs(ref[i] - exact(xs2[i])) for i in range(len(xs2))))

=== shooting, p=4 ===
       x           y(h)          y(2h)              R           y_RR          exact
             1              2              2              0              2              2
           1.2      2.4189569      2.4199581 -6.6747664e-05      2.4188901      2.4187859
           1.4      2.8712632      2.8724693 -8.0407184e-05      2.8711828      2.8710611
           1.6      3.3522374      3.3536263 -9.2592382e-05      3.3521448      3.3520058
           1.8      3.8582768      3.8598438 -0.00010446595      3.8581723       3.858016
             2      4.3865843      4.3883279 -0.00011623746      4.3864681      4.3862944
max |R| = 0.00011623746392110945
max |y_RR - exact| = 0.00017370691629192692

=== finite differences, p=2 ===
       x           y(h)          y(2h)              R           y_RR          exact
             1              2              2              0              2              2
           1.2      2.4190408      2.4197802 -0.00024647269      2.418